In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1aEJy3n95vrFoGbIbMntT7wWU_NtmQyW8", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# Notebook 2: Gradient Accumulation

**Vizuara AI Pods | GPU Programming Course | Pod 3: Recomputation, Accumulation, and Data Parallelism**

---

In this notebook, we will build a thorough understanding of **gradient accumulation** -- the technique that lets us simulate large effective batch sizes while only processing small micro-batches at a time.

By the end of this notebook, you will:

1. Understand the mathematical equivalence between large-batch training and gradient accumulation
2. Implement gradient accumulation from scratch in a clean training loop
3. Verify experimentally that accumulated gradients match true large-batch gradients
4. Measure the memory savings and training time trade-offs
5. Combine gradient accumulation with activation checkpointing

**Prerequisites:** Notebook 1 (Activation Checkpointing), basic PyTorch training loop knowledge.

**Estimated time:** 40-50 minutes

**Runtime:** Make sure you are using a GPU runtime in Colab: Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
!pip install -q torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib.pyplot as plt
import numpy as np
import time
import gc

assert torch.cuda.is_available(), "No GPU found! Please enable GPU in Runtime > Change runtime type."
device = torch.device('cuda')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
def get_memory_mb():
    return torch.cuda.memory_allocated() / (1024 ** 2)

def get_peak_memory_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2)

def reset_memory_stats():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
#@title 🎧 Listen: Why Large Batches
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_large_batches.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: Why Large Batches Matter (and Why We Cannot Just Use Them)

Large batch sizes lead to smoother gradient estimates, more stable training, and often better convergence. But large batches require proportionally more activation memory.

Let's see this memory scaling in action.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Scaling Experiment
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_memory_scaling_experiment.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SmallTransformer(nn.Module):
    """A small transformer for gradient accumulation experiments."""
    def __init__(self, vocab_size=1000, hidden_dim=512, num_layers=6, num_heads=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(512, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=4*hidden_dim,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.encoder(x)
        return self.head(x)


# Measure memory for different batch sizes
vocab_size = 1000
seq_len = 128
batch_sizes = [4, 8, 16, 32, 64]

print("Peak GPU memory for different batch sizes (6-layer transformer):")
print("=" * 55)

mem_results = []
for bs in batch_sizes:
    reset_memory_stats()
    try:
        model = SmallTransformer().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        x = torch.randint(0, vocab_size, (bs, seq_len), device=device)
        labels = torch.randint(0, vocab_size, (bs, seq_len), device=device)

        torch.cuda.reset_peak_memory_stats()
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out.view(-1, vocab_size), labels.view(-1))
        loss.backward()
        optimizer.step()

        peak_mem = get_peak_memory_mb()
        mem_results.append((bs, peak_mem))
        print(f"  Batch size {bs:>4}: {peak_mem:>8.1f} MB")

        del model, optimizer, x, labels, out, loss
        reset_memory_stats()
    except RuntimeError:
        print(f"  Batch size {bs:>4}: OOM!")
        reset_memory_stats()

print("\nMemory scales roughly linearly with batch size.")
print("Gradient accumulation solves this: use small batches, accumulate over K steps.")

In [ ]:
#@title 🎧 Listen: Math Equivalence Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_math_equivalence_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: The Mathematical Equivalence

The core insight of gradient accumulation: for a loss that is the mean over samples, the gradient of the full batch equals the mean of the micro-batch gradients.

$$\nabla_\theta \mathcal{L}_{\text{batch}} = \frac{1}{K} \sum_{k=1}^{K} \nabla_\theta \mathcal{L}_{\text{micro-batch}_k}$$

Let's prove this experimentally by computing gradients both ways and comparing.

In [ ]:
#@title 🎧 Code Walkthrough: Math Equivalence Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_math_equivalence_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Create a small model and fixed data for reproducibility
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

# Full batch: 32 samples
full_batch_x = torch.randn(32, 64, device=device)
full_batch_y = torch.randint(0, 10, (32,), device=device)

# METHOD 1: Full batch gradient (ground truth)
model.zero_grad()
out = model(full_batch_x)
loss_full = F.cross_entropy(out, full_batch_y)
loss_full.backward()

# Save the full-batch gradients
grads_full_batch = {name: p.grad.clone() for name, p in model.named_parameters()}
print(f"Full batch loss: {loss_full.item():.6f}")

# METHOD 2: Gradient accumulation with K=8 micro-batches of size 4
K = 8
micro_batch_size = 4

model.zero_grad()
accumulated_loss = 0.0

for k in range(K):
    start = k * micro_batch_size
    end = start + micro_batch_size

    micro_x = full_batch_x[start:end]
    micro_y = full_batch_y[start:end]

    out = model(micro_x)
    loss = F.cross_entropy(out, micro_y)

    # Scale by 1/K so accumulated gradient = mean of micro-batch gradients
    scaled_loss = loss / K
    scaled_loss.backward()  # Gradients ACCUMULATE (PyTorch adds to .grad)

    accumulated_loss += loss.item() / K

print(f"Accumulated loss: {accumulated_loss:.6f}")

# Compare gradients
print(f"\n{'Parameter':<30} {'Max Gradient Difference':>25}")
print("-" * 60)

all_close = True
for name, p in model.named_parameters():
    diff = (grads_full_batch[name] - p.grad).abs().max().item()
    match = diff < 1e-5
    all_close = all_close and match
    print(f"{name:<30} {diff:>25.2e}  {'OK' if match else 'MISMATCH'}")

print(f"\nAll gradients match: {all_close}")
print("Gradient accumulation is mathematically equivalent to large-batch training!")

del model
reset_memory_stats()

In [ ]:
#@title 🎧 Transition: Clean Training Loop Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_clean_training_loop_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: Clean Gradient Accumulation Training Loop

Now let's implement a proper training loop with gradient accumulation and train a model on a real task. We will create a synthetic classification dataset to keep things focused on the accumulation mechanics.

In [ ]:
#@title 🎧 Code Walkthrough: Data Model And Train Func
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_data_model_and_train_func.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Create a synthetic dataset
torch.manual_seed(42)
np.random.seed(42)

num_samples = 4096
input_dim = 128
num_classes = 10

# Generate data with some structure (not purely random)
centers = torch.randn(num_classes, input_dim) * 3
labels = torch.randint(0, num_classes, (num_samples,))
data = centers[labels] + torch.randn(num_samples, input_dim) * 0.5

dataset = TensorDataset(data, labels)
print(f"Dataset: {num_samples} samples, {input_dim} features, {num_classes} classes")

In [ ]:
class ClassificationModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=4):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)]
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)])
        layers.append(nn.Linear(hidden_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_with_accumulation(model, dataset, micro_batch_size, accumulation_steps,
                            num_epochs=5, lr=1e-3):
    """
    Train a model using gradient accumulation.

    Args:
        micro_batch_size: size of each micro-batch (what fits in memory)
        accumulation_steps: K -- number of micro-batches per optimizer step
        effective batch size = micro_batch_size * accumulation_steps
    """
    effective_batch_size = micro_batch_size * accumulation_steps
    dataloader = DataLoader(dataset, batch_size=micro_batch_size, shuffle=True, drop_last=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(dataloader) // accumulation_steps
    )

    history = {'loss': [], 'accuracy': [], 'step_times': []}
    global_step = 0

    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        micro_step = 0

        optimizer.zero_grad()

        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = F.cross_entropy(outputs, targets)

            # Scale loss by accumulation steps
            scaled_loss = loss / accumulation_steps
            scaled_loss.backward()

            # Track metrics
            epoch_loss += loss.item()
            preds = outputs.argmax(dim=1)
            epoch_correct += (preds == targets).sum().item()
            epoch_total += targets.size(0)
            micro_step += 1

            # Optimizer step every K micro-batches
            if micro_step % accumulation_steps == 0:
                # Optional: gradient clipping (common in practice)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

        avg_loss = epoch_loss / micro_step
        accuracy = epoch_correct / epoch_total * 100
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)

        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | "
              f"Accuracy: {accuracy:.1f}% | Effective BS: {effective_batch_size} | "
              f"Optimizer steps: {global_step}")

    return history

print("Training function defined.")

In [ ]:
#@title 🎧 Narration: Train Configs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_train_configs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Train with different accumulation settings
# All three configurations have the same effective batch size of 64

configs = [
    {"micro_bs": 64, "accum_steps": 1,  "label": "BS=64 (no accumulation)"},
    {"micro_bs": 16, "accum_steps": 4,  "label": "BS=16, K=4 (effective 64)"},
    {"micro_bs": 4,  "accum_steps": 16, "label": "BS=4, K=16 (effective 64)"},
]

all_histories = []
all_peak_mems = []

for config in configs:
    print(f"\nTraining: {config['label']}")
    print("-" * 60)

    torch.manual_seed(42)
    model = ClassificationModel(input_dim, hidden_dim=256, num_classes=num_classes).to(device)

    reset_memory_stats()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    history = train_with_accumulation(
        model, dataset,
        micro_batch_size=config['micro_bs'],
        accumulation_steps=config['accum_steps'],
        num_epochs=5, lr=1e-3
    )
    elapsed = time.perf_counter() - start_time
    peak_mem = get_peak_memory_mb()

    print(f"  Total time: {elapsed:.1f}s | Peak memory: {peak_mem:.1f} MB")
    all_histories.append((config['label'], history))
    all_peak_mems.append((config['label'], peak_mem))

    del model
    reset_memory_stats()

In [ ]:
#@title 🎧 What to Look For: Visualize Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_visualize_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize training curves and memory comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#E53935', '#1E88E5', '#43A047']

# Plot 1: Loss curves
ax = axes[0]
for (label, hist), color in zip(all_histories, colors):
    ax.plot(hist['loss'], 'o-', color=color, linewidth=2, markersize=6, label=label)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curves
ax = axes[1]
for (label, hist), color in zip(all_histories, colors):
    ax.plot(hist['accuracy'], 'o-', color=color, linewidth=2, markersize=6, label=label)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 3: Peak memory comparison
ax = axes[2]
labels_short = ['BS=64\n(no accum)', 'BS=16\nK=4', 'BS=4\nK=16']
mems = [m for _, m in all_peak_mems]
bars = ax.bar(labels_short, mems, color=colors, alpha=0.8, width=0.5)
for bar, mem in zip(bars, mems):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{mem:.0f} MB', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Peak Memory (MB)', fontsize=12)
ax.set_title('Peak GPU Memory', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. All three configurations converge to similar loss/accuracy")
print("   (because they have the same effective batch size)")
print("2. Smaller micro-batches use significantly less memory")
print("3. The trade-off: more accumulation steps = longer wall-clock time")

In [ ]:
#@title 🎧 Transition: Pitfalls Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_pitfalls_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Common Pitfalls and Best Practices

Gradient accumulation is simple in concept, but there are several common mistakes. Let's explore them.

In [ ]:
#@title 🎧 Code Walkthrough: Pitfall1 Loss Scaling
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_pitfall1_loss_scaling.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# PITFALL 1: Forgetting to scale the loss
# If you don't divide by K, the accumulated gradient is K times too large!

torch.manual_seed(42)
model = nn.Linear(64, 10).to(device)
x = torch.randn(32, 64, device=device)
y = torch.randint(0, 10, (32,), device=device)

# Correct: scale loss by 1/K
K = 4
micro_bs = 8

model.zero_grad()
for k in range(K):
    out = model(x[k*micro_bs:(k+1)*micro_bs])
    loss = F.cross_entropy(out, y[k*micro_bs:(k+1)*micro_bs]) / K  # <-- Correct!
    loss.backward()
grad_correct = model.weight.grad.clone()

# WRONG: no loss scaling
model.zero_grad()
for k in range(K):
    out = model(x[k*micro_bs:(k+1)*micro_bs])
    loss = F.cross_entropy(out, y[k*micro_bs:(k+1)*micro_bs])  # <-- WRONG! Not scaled
    loss.backward()
grad_wrong = model.weight.grad.clone()

# Reference: full batch
model.zero_grad()
out = model(x)
loss = F.cross_entropy(out, y)
loss.backward()
grad_ref = model.weight.grad.clone()

print("Gradient norms:")
print(f"  Full batch (reference):  {grad_ref.norm().item():.4f}")
print(f"  Accumulated (correct):   {grad_correct.norm().item():.4f}")
print(f"  Accumulated (WRONG):     {grad_wrong.norm().item():.4f}")
print(f"\n  Ratio (wrong/correct): {grad_wrong.norm().item() / grad_correct.norm().item():.1f}x")
print(f"  This equals K = {K}! Without scaling, gradients are K times too large.")

del model
reset_memory_stats()

In [ ]:
#@title 🎧 Code Walkthrough: Pitfall2 Zero Grad
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_pitfall2_zero_grad.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# PITFALL 2: Forgetting to zero gradients at the right time
#
# The rule: zero_grad() should be called AFTER optimizer.step(),
# not before every backward() call.

print("Gradient Accumulation Checklist:")
print("=" * 50)
print()
print("1. optimizer.zero_grad()        -- at the START of each accumulation cycle")
print("2. for k in range(K):")
print("     loss = compute_loss(micro_batch_k)")
print("     (loss / K).backward()       -- scale and accumulate")
print("3. optimizer.step()              -- update weights with accumulated gradient")
print("4. optimizer.zero_grad()         -- reset for next cycle")
print()
print("Common mistake: calling optimizer.zero_grad() inside the loop!")
print("This would erase previously accumulated gradients.")

In [ ]:
#@title 🎧 Code Walkthrough: Pitfall3 Batchnorm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_pitfall3_batchnorm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# PITFALL 3: BatchNorm and gradient accumulation
#
# BatchNorm computes running statistics from each micro-batch.
# With small micro-batches, these statistics are noisy.
# Solution: use LayerNorm (which is per-sample) or GroupNorm.

torch.manual_seed(42)

# Demonstrate the problem
bn = nn.BatchNorm1d(64).to(device)
ln = nn.LayerNorm(64).to(device)

# Large batch statistics
x_large = torch.randn(32, 64, device=device)
bn_large = bn(x_large)

# Tiny micro-batch statistics (noisy!)
bn.reset_running_stats()
x_tiny = x_large[:2]  # only 2 samples
bn_tiny = bn(x_tiny)

print("BatchNorm running mean difference (large vs tiny batch):")
print(f"  Mean abs difference: {(bn.running_mean).abs().mean().item():.4f}")
print()
print("Recommendation: Use LayerNorm instead of BatchNorm when using")
print("gradient accumulation with small micro-batches.")
print("(Transformers already use LayerNorm, so this is usually not an issue.)")

del bn, ln
reset_memory_stats()

In [ ]:
#@title 🎧 Transition: Combine Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_combine_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: Combining Checkpointing + Accumulation

The real power comes from combining these techniques. Activation checkpointing reduces per-micro-batch memory, and gradient accumulation allows us to simulate large effective batches.

Let's see them work together.

In [ ]:
#@title 🎧 Code Walkthrough: Combine Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_combine_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
from torch.utils.checkpoint import checkpoint as ckpt

class CheckpointedSmallTransformer(nn.Module):
    """Small transformer with activation checkpointing."""
    def __init__(self, vocab_size=1000, hidden_dim=512, num_layers=6, num_heads=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(512, hidden_dim)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=num_heads, dim_feedforward=4*hidden_dim,
                batch_first=True, activation='gelu'
            ) for _ in range(num_layers)
        ])
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        for layer in self.layers:
            x = ckpt(layer, x, use_reentrant=False)
        return self.head(x)


# Compare four configurations:
# 1. No optimization (large micro-batch)
# 2. Checkpointing only
# 3. Accumulation only
# 4. Both checkpointing + accumulation

configs = [
    {"ckpt": False, "micro_bs": 32, "accum": 1,  "label": "No optimization (BS=32)"},
    {"ckpt": True,  "micro_bs": 32, "accum": 1,  "label": "Checkpointing only (BS=32)"},
    {"ckpt": False, "micro_bs": 8,  "accum": 4,  "label": "Accumulation only (BS=8, K=4)"},
    {"ckpt": True,  "micro_bs": 8,  "accum": 4,  "label": "Both (BS=8, K=4, checkpoint)"},
]

print("Combined techniques -- memory comparison (effective BS=32 for all):")
print("=" * 75)

combined_results = []
vocab_size = 1000
seq_len = 128

for cfg in configs:
    reset_memory_stats()
    try:
        if cfg['ckpt']:
            model = CheckpointedSmallTransformer(vocab_size=vocab_size).to(device)
        else:
            model = SmallTransformer(vocab_size=vocab_size).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        torch.cuda.reset_peak_memory_stats()

        optimizer.zero_grad()
        for k in range(cfg['accum']):
            x = torch.randint(0, vocab_size, (cfg['micro_bs'], seq_len), device=device)
            labels = torch.randint(0, vocab_size, (cfg['micro_bs'], seq_len), device=device)
            out = model(x)
            loss = F.cross_entropy(out.view(-1, vocab_size), labels.view(-1))
            (loss / cfg['accum']).backward()

        optimizer.step()
        peak_mem = get_peak_memory_mb()
        combined_results.append((cfg['label'], peak_mem))
        print(f"  {cfg['label']:<50} Peak: {peak_mem:>8.1f} MB")

        del model, optimizer
        reset_memory_stats()
    except RuntimeError as e:
        print(f"  {cfg['label']:<50} OOM!")
        combined_results.append((cfg['label'], None))
        reset_memory_stats()

In [ ]:
#@title 🎧 What to Look For: Combine Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_combine_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the combined results
labels_viz = [l.replace(' (', '\n(') for l, _ in combined_results if _ is not None]
mems_viz = [m for _, m in combined_results if m is not None]
colors_viz = ['#E53935', '#FF9800', '#1E88E5', '#43A047'][:len(mems_viz)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(mems_viz)), mems_viz, color=colors_viz, alpha=0.85, width=0.6)
ax.set_xticks(range(len(mems_viz)))
ax.set_xticklabels(labels_viz, fontsize=9, ha='center')
ax.set_ylabel('Peak GPU Memory (MB)', fontsize=12)
ax.set_title('Memory Usage: Combined Optimization Techniques', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, mem in zip(bars, mems_viz):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{mem:.0f} MB', ha='center', fontsize=11, fontweight='bold')

if len(mems_viz) >= 4:
    savings = (1 - mems_viz[3] / mems_viz[0]) * 100
    ax.text(0.95, 0.95, f'Combined savings: {savings:.0f}%',
            transform=ax.transAxes, fontsize=12, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='#43A047', alpha=0.3))

plt.tight_layout()
plt.show()

print("\nCheckpointing + Accumulation together provide the most memory savings!")
print("This is the standard setup for training large models on limited hardware.")

In [ ]:
#@title 🎧 Narration: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Exercises

### TODO Exercise 1: Verify Gradient Equivalence with Different K Values

We showed that K=8 gives the same gradients as a full batch. Verify this holds for K=2, 4, 16, and 32. Plot the maximum gradient error as a function of K.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Verify gradient equivalence for multiple K values
#
# Steps:
# 1. Create a model and a fixed batch of 64 samples
# 2. Compute the reference full-batch gradient
# 3. For K in [2, 4, 8, 16, 32]:
#    a. Split the batch into K micro-batches of size 64/K
#    b. Accumulate gradients with proper scaling
#    c. Compute max absolute difference from reference
# 4. Plot the error vs K
#
# Expected result: error should be ~0 (< 1e-5) for all K values

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

total_samples = 64
x = torch.randn(total_samples, 64, device=device)
y = torch.randint(0, 10, (total_samples,), device=device)

# Reference gradient
model.zero_grad()
# TODO: Compute full-batch gradient
# ref_grad = ...

K_values = [2, 4, 8, 16, 32]
errors = []

for K in K_values:
    micro_bs = total_samples // K
    model.zero_grad()
    # TODO: Accumulate gradients over K micro-batches
    # TODO: Compute max error vs reference
    # errors.append(max_error)
    pass

# TODO: Plot errors vs K
# All errors should be essentially zero (floating point noise only)

In [ ]:
#@title 🎧 Narration: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Learning Rate Scaling with Gradient Accumulation

When you increase the effective batch size, you typically need to increase the learning rate proportionally (the "linear scaling rule"). Verify this by training the same model with:

- `micro_bs=8, K=1, lr=1e-3` (effective BS = 8)
- `micro_bs=8, K=4, lr=1e-3` (effective BS = 32, same LR -- may not converge as well)
- `micro_bs=8, K=4, lr=4e-3` (effective BS = 32, scaled LR)

Compare the training curves.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Learning rate scaling experiment
#
# Steps:
# 1. Use the ClassificationModel and dataset from Part 3
# 2. Train three configurations:
#    a. Baseline: micro_bs=8, K=1, lr=1e-3
#    b. Large batch, same LR: micro_bs=8, K=4, lr=1e-3
#    c. Large batch, scaled LR: micro_bs=8, K=4, lr=4e-3
# 3. Plot all three loss curves
#
# Expected: configuration (c) should match configuration (a) best

# configs = [
#     {"micro_bs": 8, "accum": 1, "lr": 1e-3, "label": "BS=8, LR=1e-3"},
#     {"micro_bs": 8, "accum": 4, "lr": 1e-3, "label": "BS=32, LR=1e-3 (unscaled)"},
#     {"micro_bs": 8, "accum": 4, "lr": 4e-3, "label": "BS=32, LR=4e-3 (scaled)"},
# ]
#
# for cfg in configs:
#     torch.manual_seed(42)
#     model = ClassificationModel(input_dim, 256, num_classes).to(device)
#     history = train_with_accumulation(
#         model, dataset,
#         micro_batch_size=cfg['micro_bs'],
#         accumulation_steps=cfg['accum'],
#         num_epochs=10, lr=cfg['lr']
#     )
#     # Plot results...

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Proved mathematical equivalence** -- accumulated micro-batch gradients exactly equal full-batch gradients

2. **Implemented a clean training loop** -- with proper loss scaling (`loss / K`) and gradient clipping

3. **Verified convergence** -- three different accumulation settings (same effective BS) achieve the same accuracy

4. **Identified common pitfalls** -- loss scaling, zero_grad timing, and BatchNorm issues

5. **Combined with checkpointing** -- the two techniques compose naturally for maximum memory savings

### Key Takeaways

- Gradient accumulation is mathematically exact (not an approximation)
- `effective_batch_size = micro_batch_size * accumulation_steps`
- Always scale the loss by `1/K` before `.backward()`
- The only cost is wall-clock time (K sequential forward-backward passes)
- Combine with activation checkpointing for maximum memory efficiency
- Scale learning rate linearly when increasing effective batch size

### Next Notebook

In Notebook 3, we will explore **data parallelism** -- the technique that distributes training across multiple GPUs, each processing a different shard of the data with synchronized gradient updates via AllReduce.